# Reference notebook: developing your own model

This notebook shows how the project builds a VaR model **end to end** on a small example (4 stocks, the last few months) and ends with a loop for trying your own model. First we walk through one window (one *calendar*) by hand, then apply the same thing to all dates (*walk-forward*) in a few lines.

**Flow**

```
prices ──► Calendar (train / forecast days, portfolio return)
               │
               ├─► FeatureSpec + Target ──► build_features ──► training / forecast matrices
               │
               ├─► xgboost_train ──► model + (ResidualBank, EwmaVarianceBank)   ← residuals of the training days
               │
               └─► calculate_multi_day_standart_residuals_… ──► residuals of the forecast days
                                                                      │
                                                    var_from_residual_bank_… ──► VaR  ──► backtest
```

**Concepts (in short)**
- **Calendar:** one training window and the forecast window that follows it. It carries the portfolio return and loss (`portfolio_loss = −return`).
- **Long / wide:** in a *long* model every ticker-day is one row and a single pooled model forecasts the ticker variance; the portfolio variance is then built as `w'Σw`. In a *wide* model a row is one day and the target is the portfolio's variance itself.
- **ResidualBank:** keeps the standardized residual `(return − μ̂) / σ̂` of every day. **VaR = −(μ̂ + σ̂·q)**, where q is the (1 − confidence) quantile of the previous N residuals: *filtered historical simulation*. Banks are **single-use**: a new one is created for every training.
- **Unit:** `"percent"` (simple return) or `"log"`; every record is kept in this unit, while the readings (VaR, loss) are always shown as percent returns.

In [1]:
# the notebook lives in core/: move to the project root (the code and the relative data paths are there)
import os
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "core" / "__init__.py").exists())
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import contextlib
import io
import tempfile
import warnings

warnings.filterwarnings("ignore")
import pandas as pd

import core
from core import feature_blocks
from core import run_store as rs
from core import walk_forward as wf
from core.models import tree_models as tm
from core.targets import RatioToBase, SquaredReturn
from tools.graph_helpers import plot_var_models_vs_loss, plot_var_vs_loss

pd.set_option("display.width", 200)

## 0. Data

Raw **prices** (the calendar computes the returns itself). This example uses the small 4-stock file; the full data is in `data_general/`.

In [2]:
TICKERS = ["AAPL", "JPM", "XOM", "KO"]
WEIGHTS = {"AAPL": 0.25, "JPM": 0.25, "XOM": 0.25, "KO": 0.25}      # normalised to sum 1
UNIT = "percent"                                                       # the return type of the records: "percent" | "log"

prices = pd.read_csv("data_general/4_ticker_data.csv", parse_dates=["Date"], index_col="Date")[TICKERS]
prices.tail(3)

,AAPL,JPM,XOM,KO
Date,,,,
2024-12-27,254.201370,233.912888,102.030319,60.246071
2024-12-30,250.829773,232.118561,101.340408,59.840893
2024-12-31,249.059464,232.496826,103.074760,60.062771


## 1. A calendar: training window + forecast window

`warmup_bars`: days before the training starts that are used to warm up the features (21-day windows, EWMA); they are not trained on.

In [3]:
calendar = core.DailyGroundTruthCalendar(
    prices, TICKERS, WEIGHTS,
    train_start="2024-06-03", train_end="2024-11-29",
    forecast_start="2024-12-02", forecast_end="2024-12-31",
    unit=UNIT, warmup_bars=60,
)
print(f"train: {len(calendar.train_data)} days | forecast: {len(calendar.forecast_dates)} days")
calendar.forecast_data[TICKERS + ["portfolio_return", "portfolio_loss"]].head()

[WARNING] forecast_end=2024-12-31 falls on Tuesday, expected Friday (using it as-is).
[WARNING] [train] 4 weekday days had no data and were skipped (likely market holidays): 2024-06-19, 2024-07-04, 2024-09-02, 2024-11-28
[INFO] train set: 126 bars, 4 weekday days skipped
[WARNING] [forecast] 1 weekday day had no data and were skipped (likely market holidays): 2024-12-25
[INFO] forecast set: 21 bars, 1 weekday day skipped
train: 126 days | forecast: 21 days


,AAPL,JPM,XOM,KO,portfolio_return,portfolio_loss
Date,,,,,,
2024-12-02,0.009523,-0.013896,-0.000932,-0.006710,-0.003004,0.003004
2024-12-03,0.012772,-0.005807,-0.001527,-0.002042,0.000849,-0.000849
2024-12-04,0.001484,-0.005800,-0.028809,-0.020624,-0.013437,0.013437
2024-12-05,0.000123,0.008546,0.004375,0.011413,0.006114,-0.006114
2024-12-06,-0.000823,0.007659,-0.010542,-0.006198,-0.002476,0.002476


## 2. Features and target

- A **`FeatureSpec`** is a list of feature *blocks* (return, |return|, 5/10/21-day statistics, EWMA vol, ...). `FeatureSpec.default(unit=...)` is the project's default set; `.with_blocks(...)` adds a block.
- A **`Target`** says what the model forecasts: `SquaredReturn` (the next day's r², with a scale), `RatioToBase` (r² / EWMA variance), `RealizedVariance`, `RelativeRvChange`, ... (the RV targets need `additional_feature_data`, which this small example does not have).
- `core.build_features(calendar, "long" | "wide", spec, target)` builds the training and the forecast matrices. Note that the target is the **next day**: the features of a day only use what is known up to that day (no look-ahead).

In [4]:
spec = core.FeatureSpec.default(unit=UNIT)
target = SquaredReturn(10000, unit=UNIT)        # r² · 10000: only a scale, to keep the numbers handy

training_feature, forecast_feature = core.build_features(calendar, "long", spec, target)
print("feature columns:", list(training_feature.feature_cols))
print("training rows:", len(training_feature.X_in_sample), "| forecast rows:", len(forecast_feature.X_forecast_sample))
training_feature.X_in_sample.head(3)

[INFO] long train: 504 rows -> 504 after dropna (0 dropped, 0.0%)
[INFO] long forecast: 88 rows -> 88 after dropna (0 dropped, 0.0%)
feature columns: ['r', 'abs_r', 'real_vol_5', 'mean_abs_5', 'skew_5', 'real_vol_10', 'mean_abs_10', 'skew_10', 'real_vol_21', 'mean_abs_21', 'skew_21', 'ewma_vol', 'market_ewma_vol']
training rows: 500 | forecast rows: 88


,r,abs_r,real_vol_5,mean_abs_5,skew_5,real_vol_10,mean_abs_10,skew_10,real_vol_21,mean_abs_21,skew_21,ewma_vol,market_ewma_vol
0,0.009259,0.009259,0.003586,0.004233,0.332333,0.010286,0.007937,-1.278711,0.015262,0.009890,2.104064,0.012072,0.008579
1,-0.003998,0.003998,0.010079,0.007942,1.118287,0.019093,0.013715,-1.183706,0.014391,0.010424,-1.672663,0.014336,0.008579
2,0.000000,0.000000,0.007520,0.004942,1.563111,0.007935,0.005477,0.208281,0.006141,0.004560,-0.265555,0.007053,0.008579


## 3. Train the long model

`xgboost_train` fits the model and writes the residuals of the **training days** into the banks. Banks are single-use, so every training opens new ones. `ResidualBank(mu_mode="constant", default_mu_hat=0.0)`: the mean return is taken as zero (with `"rolling"` the mean of the last N days is used).

In [5]:
residual_bank = core.ResidualBank(mu_mode="constant", default_mu_hat=0.0)
ewma_bank = core.EwmaVarianceBank(calendar)             # long only: the EWMA covariance (for the correlation)

model = tm.xgboost_train(calendar, training_feature, residual_bank, ewma_bank=ewma_bank)
print("residuals of the training days:", len(model.residual_bank_in_sample.residuals))
model.residual_bank_in_sample.residuals.tail(3)

[INFO] train_dates_past_burn_in: 125/125 candidate dates are past the burn-in (burn_in_bars=29, warmup_bars=60, train_data starts 2024-06-03, burn-in ends 2024-04-18, 0 date(s) dropped).
residuals of the training days: 125


,actual_return,predicted_variance,predicted_sigma,mu_hat,standardized_residual
Date,,,,,
2024-11-26,-0.001476,0.000036,0.005986,0.0,-0.246585
2024-11-27,-0.00144,0.00004,0.006304,0.0,-0.228436
2024-11-29,0.003649,0.00003,0.005499,0.0,0.66362


## 4. The forecast window and the VaR

`calculate_multi_day_standart_residuals_from_long_format_model` applies the model to the forecast days and adds their residuals to a **copy** of the bank (the bank of the training stays as it is). *How the VaR is read* (confidence, quantile window, minimum days) belongs to the reading call, not to the bank: the same bank can be read again in any way you like.

In [6]:
bank_long, ewma_bank_long = model.calculate_multi_day_standart_residuals_from_long_format_model(calendar, forecast_feature)

VAR_CONFIDENCE = 0.95
VAR_WINDOW = 60          # the quantile of the last 60 residuals (in this small example; the project usually uses 252)
VAR_MIN_PERIODS = 30

var_long = calendar.var_from_residual_bank_forecast_period(
    bank_long, VAR_CONFIDENCE, VAR_WINDOW, multi_day_var_min_periods=VAR_MIN_PERIODS,
)
var_long.head()

,varForecast,n_residuals_used,loss
Date,,,
2024-12-02,0.007449,60,0.003004
2024-12-03,0.009209,60,-0.000849
2024-12-04,0.007514,60,0.013437
2024-12-05,0.017992,60,-0.006114
2024-12-06,0.008291,60,0.002476


In [7]:
plot_var_vs_loss(var_long, bank_long, calendar=calendar)        # the VaR (line), the realized loss, the exceedances marked

Figure

## 5. Backtest

`run_full_backtest_suite_single_window` gives the number of exceedances and three tests: **Kupiec** (is the exceedance rate right), **Christoffersen independence** (do the exceedances cluster) and **conditional coverage** (both together). `run_level_and_volatility_suite_single_window` gives the level of the VaR and its **tick loss** (the loss from exceedances plus the loss from unused capital); lower is better.

In [8]:
backtest = core.run_full_backtest_suite_single_window(var_long, var_long.index[0], var_long.index[-1], VAR_CONFIDENCE, 0.95)
level = core.run_level_and_volatility_suite_single_window(var_long, VAR_CONFIDENCE)
pd.Series({**backtest, **level})[["n_days", "n_exceedances", "exceedance_rate", "kupiec_pof_pass", "christoffersen_ind_pass", "mean_tick_loss", "mean_var"]]

n_days                           21
n_exceedances                     2
exceedance_rate            0.095238
kupiec_pof_pass                True
christoffersen_ind_pass        True
mean_tick_loss             0.001216
mean_var                   0.010279
dtype: object

## 6. The wide model (same calendar)

The same chain, but the target is the **portfolio's** variance directly: there are no per-ticker forecasts and no EWMA bank.

In [9]:
training_wide, forecast_wide = core.build_features(calendar, "wide", spec, target)

model_wide = tm.xgboost_train(calendar, training_wide, core.ResidualBank(mu_mode="constant", default_mu_hat=0.0))
bank_wide = model_wide.calculate_multi_day_standart_residuals_from_wide_format_model(calendar, forecast_wide)
var_wide = calendar.var_from_residual_bank_forecast_period(bank_wide, VAR_CONFIDENCE, VAR_WINDOW, multi_day_var_min_periods=VAR_MIN_PERIODS)

plot_var_models_vs_loss({"xgboost_long": (var_long, bank_long), "xgboost_wide": (var_wide, bank_wide)}, calendar=calendar)

[INFO] wide train: 126 bars -> 126 after dropna (0 dropped, 0.0%)
[INFO] wide forecast: 22 bars -> 22 after dropna (0 dropped, 0.0%)


Figure

## 7. Walk-forward: the same thing over all dates

One calendar is one window. A **walk-forward** moves the windows forward one after the other: the model is trained again in every window, the forecast days are put one after the other, and the result is **one VaR series**.

- `WalkForwardSetting`: the data, the tickers, the weights, the number of training / forecast days. `window_table()` shows the windows.
- `ModelRecipe`: the recipe of a model (long / wide, features, target, hyperparameters). The baselines are recipes too: `wf.BASELINE_RECIPES` (GARCH, EWMA, historical).
- `train_walk_forward(calendars, recipe)` returns one `fit` per window; `create_combined_residuals_bank_and_read_var_df(fits, confidence, window, minimum)` reads all of them into one VaR series.

In [10]:
setting = wf.WalkForwardSetting(prices, TICKERS, WEIGHTS, train_bars=252, forecast_bars=63, unit=UNIT, warmup_bars=60)
print(len(setting.window_table()), "windows")
setting.window_table()[["train_start", "train_end", "forecast_start", "forecast_end"]].head(3)

15 windows


,train_start,train_end,forecast_start,forecast_end
0,2020-03-31,2021-03-30,2021-03-31,2021-06-29
1,2020-06-30,2021-06-29,2021-06-30,2021-09-28
2,2020-09-29,2021-09-28,2021-09-29,2021-12-28


In [11]:
with contextlib.redirect_stdout(io.StringIO()):        # the calendar prints a warning for every market holiday; silenced here
    calendars = setting.make_calendars()

recipes = {
    "xgb_long": wf.ModelRecipe("long", features=spec, target=target),
    **wf.BASELINE_RECIPES,                              # garch_wide, ewma, historical
}
fits = {}
for name, recipe in recipes.items():
    with contextlib.redirect_stdout(io.StringIO()):
        fits[name] = wf.train_walk_forward(calendars, recipe)
list(fits)

['xgb_long', 'garch_wide', 'ewma', 'historical']

In [12]:
WINDOW, MIN_PERIODS = 126, 30
var_dfs = {name: wf.create_combined_residuals_bank_and_read_var_df(f, VAR_CONFIDENCE, WINDOW, MIN_PERIODS) for name, f in fits.items()}

ticks = core.daily_tick_losses(var_dfs, VAR_CONFIDENCE).mean()        # on the same days, with the same realized loss
pd.DataFrame({
    "mean tick loss": ticks,
    "vs garch %": (ticks / ticks["garch_wide"] - 1) * 100,
    "exceedances": {n: int((v["loss"] > v["varForecast"]).sum()) for n, v in var_dfs.items()},
}).sort_values("mean tick loss").round(5)

,mean tick loss,vs garch %,exceedances
historical,0.00111,-1.49107,49
ewma,0.00111,-1.10922,54
garch_wide,0.00112,0.00000,57
xgb_long,0.00118,5.39502,52


A word of caution: in this small example (4 stocks, about 1,000 days) do not read a "winner" into a single model or baseline; the differences are inside the noise. For significance use `core.select_best_run(var_dfs, 0.95, 0.95, baseline="garch_wide")` (Diebold-Mariano, Model Confidence Set); for the research results see `notebooks/analysis_report.ipynb`.

## 8. Save and load

Saving a run is what lets it enter the dashboard and the comparison tools. Here we save into a **temporary** folder (nothing is written to the real `store/`). For a real save:

```python
run = rs.train_and_save_walk_forward_run(setting, recipe, "my_xgb_long")    # writes under store/; the same recipe is not trained again
rs.log_walk_forward_run_to_mlflow(run, wf.Reading(), experiment="walk_forward_horizon")   # the MLflow index (the dashboard lists runs from it)
```

In [13]:
temp_store = Path(tempfile.mkdtemp())
path = rs.save_walk_forward_run(setting, recipes["xgb_long"], fits["xgb_long"], "my_xgb_long", root=temp_store, slim=True)
run = wf.LoadedRun(path)

reading = wf.Reading(var_confidence=0.95, window_size=126, min_periods=30, test_confidence=0.95)
print(run.display_name, "|", len(run.var_df(reading)), "forecast days")
run.var_df(reading).head(3)

my_xgb_long | train 252 f 63 | 945 forecast days


,varForecast,n_residuals_used,loss,window_id
Date,,,,
2021-03-31,0.015322,126,0.004812,0
2021-04-01,0.014652,126,-0.010207,0
2021-04-05,0.013397,126,-0.004771,0


## 9. Try your own model

An experiment is **changing a recipe** and running the same comparison again. Three typical changes below; add yours to `candidates` and run the cell.

In [14]:
candidates = {
    # 1) another target: r² divided by the EWMA variance instead of the squared return
    "xgb_long_ratio": wf.ModelRecipe("long", features=spec, target=RatioToBase(clip=20.0, unit=UNIT)),
    # 2) hyperparameters: model_kwargs go straight to xgboost
    "xgb_long_shallow": wf.ModelRecipe("long", features=spec, target=target, model_kwargs=dict(n_estimators=150, max_depth=3)),
    # 3) one more feature: the ticker one-hot (tells the pooled model which ticker a row is)
    "xgb_long_ticker": wf.ModelRecipe("long", features=spec.with_blocks(feature_blocks.TickerOneHot()), target=target),
}
for name, recipe in candidates.items():
    with contextlib.redirect_stdout(io.StringIO()):
        fits[name] = wf.train_walk_forward(calendars, recipe)
    var_dfs[name] = wf.create_combined_residuals_bank_and_read_var_df(fits[name], VAR_CONFIDENCE, WINDOW, MIN_PERIODS)

ticks = core.daily_tick_losses(var_dfs, VAR_CONFIDENCE).mean()
pd.DataFrame({"mean tick loss": ticks, "vs garch %": (ticks / ticks["garch_wide"] - 1) * 100}).sort_values("mean tick loss").round(5)

,mean tick loss,vs garch %
historical,0.00111,-1.49107
ewma,0.00111,-1.10922
garch_wide,0.00112,0.00000
xgb_long_ratio,0.00114,1.94055
xgb_long_ticker,0.00118,5.29614
xgb_long,0.00118,5.39502
xgb_long_shallow,0.00120,7.08702


### Where to change what

| What you want | Where to look |
|---|---|
| A new **feature** | `core/feature_blocks.py`: write a `FeatureBlock` subclass (`compute` returns the columns) and add it with `FeatureSpec.with_blocks(...)`. For outside data (VIX, RV, yields) use `AdditionalFeatureBlock` and the CSVs in `data_general/` |
| A new **target** | `core/targets.py`: a `Target` subclass; how the prediction becomes a variance is defined there |
| Another **model** | `core/models/tree_models.py` (`xgboost_train`, `random_forest_train`), `core/models/garch_models.py`; same signature: `(calendar, training_feature, residual_bank, ...)` |
| Window, weights, unit | `WalkForwardSetting` (`core/walk_forward.py`); saving and loading: `core/run_store.py` |
| An **experiment script** | `training/` (examples: `train_23_experiment.py`, `train_random_weights_experiment.py`); `python training/<script>.py` |
| **Download data** (prices, 5-minute bars, RV) | `downloads/download_prices_yahoo.py`, `downloads/alpaca_calculate_rv.py` |
| Maintenance of saved runs | `tools/` (`verify_run_classes.py`, `backfill_first_window_training_bank.py`) |
| Plots (VaR, exceedance days, EWMA bank) | `tools/graph_helpers.py` |
| Apply a model to **another portfolio** (without retraining) | `ModelRecipe(..., pretrained_from=<run_id>)`; the model must have no portfolio-dependent feature (`MarketEwmaVol`): use `FeatureSpec.portable()` |
| Read the same forecasts with **other weights** | `wf.ReweightedVirtualRun(run, weights)`: nothing is trained |
| Compare, statistics | `core.select_best_run`, `core.compare_to_baseline`; dashboard: `streamlit run streamlit/dashboard.py` |
| Research results | `notebooks/analysis_report.ipynb` (`report_tables.py`) |